# Database overview
Summarises table sizes, field coverage, top work types and polity coverage of the Cultura V2 database.

In [1]:
import random
import tempfile
import numpy as np
import polars as pl
import duckdb
from style import *

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DB_PATH = '../data/cultura/humans_clean_v2.duckdb'

pl.Config.set_tbl_rows(30)
pl.Config.set_fmt_str_lengths(60)

apply_style()

## Connect

In [2]:
con = duckdb.connect(DB_PATH, read_only=True)
con.sql(f"SET memory_limit='4GB'; SET threads=6; SET temp_directory='{tempfile.mkdtemp()}';")

## Load data
### Field coverage

In [3]:
field_coverage_wide = con.sql('''
    SELECT
        COUNT(*)                                                   AS total_individuals,
        COUNT(*) FILTER (WHERE len(i.birth_date) > 0)              AS date_of_birth,
        COUNT(*) FILTER (WHERE len(i.death_date) > 0)              AS date_of_death,
        COUNT(*) FILTER (WHERE len(i.floruit_date) > 0)            AS floruit_date,
        COUNT(i.place_of_birth.entity.qid)                         AS place_of_birth,
        COUNT(i.place_of_death.entity.qid)                         AS place_of_death,
        COUNT(*) FILTER (WHERE len(i.country_of_citizenship) > 0)  AS nationality,
        COUNT(*) FILTER (WHERE len(i.occupation) > 0)              AS occupation,
        COUNT(i.sex_or_gender)                                     AS gender,
        COUNT(*) FILTER (WHERE len(i.writing_language) > 0)        AS writing_language,
        COUNT(*) FILTER (WHERE len(i.sitelink) > 0)                AS wikimedia_sitelinks,
        COUNT(*) FILTER (WHERE len(i.external_id) > 0)             AS external_identifiers,
        COUNT(e.peak_productivity.start_year)                      AS peak_productivity_window,
        COUNT(*) FILTER (WHERE e.polity_count > 0)                 AS cliopatria_polity_mapping
    FROM individual AS i
    JOIN individual_enriched AS e
      ON e.entity.qid = i.entity.qid
''').pl()

print('shape:', field_coverage_wide.shape)
field_coverage_wide

shape: (1, 14)


total_individuals,date_of_birth,date_of_death,floruit_date,place_of_birth,place_of_death,nationality,occupation,gender,writing_language,wikimedia_sitelinks,external_identifiers,peak_productivity_window,cliopatria_polity_mapping
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
13002897,7490137,3532432,70521,3724787,1585483,5570031,9038108,8965424,225825,4959581,8738235,9709528,5158972


## Preprocessing

### Field coverage table

In [4]:
PROPERTY_LABELS = {
    'total_individuals': 'Total individuals',
    'date_of_birth': 'Date of birth',
    'date_of_death': 'Date of death',
    'floruit_date': 'Floruit date',
    'place_of_birth': 'Place of birth',
    'place_of_death': 'Place of death',
    'nationality': 'Nationality',
    'occupation': 'Occupation',
    'gender': 'Gender',
    'writing_language': 'Writing language',
    'wikimedia_sitelinks': 'Wikimedia sitelinks (>=1)',
    'external_identifiers': 'External identifiers (>=1)',
    'peak_productivity_window': 'Peak-productivity window (computed)',
    'cliopatria_polity_mapping': 'Cliopatria polity mapping',
}

total_individuals = field_coverage_wide['total_individuals'].item()

def to_long(wide, key, value):
    return (
        wide.transpose(include_header=True, header_name=key, column_names=[value])
        .with_columns((pl.col(value).cast(pl.Float64) / total_individuals * 100).round(1).alias('% of total'))
    )

field_coverage = to_long(field_coverage_wide, 'Property', 'Individuals').with_columns(pl.col('Property').replace(PROPERTY_LABELS))
assert field_coverage['% of total'].max() <= 100
field_coverage.head()

Property,Individuals,% of total
str,i64,f64
"""Total individuals""",13002897,100.0
"""Date of birth""",7490137,57.6
"""Date of death""",3532432,27.2
"""Floruit date""",70521,0.5
"""Place of birth""",3724787,28.6


## Tables

### Data extracted at the beginning of the pipeline

In [5]:
field_coverage

Property,Individuals,% of total
str,i64,f64
"""Total individuals""",13002897,100.0
"""Date of birth""",7490137,57.6
"""Date of death""",3532432,27.2
"""Floruit date""",70521,0.5
"""Place of birth""",3724787,28.6
"""Place of death""",1585483,12.2
"""Nationality""",5570031,42.8
"""Occupation""",9038108,69.5
"""Gender""",8965424,68.9
